# 01. Exploratory Data Analysis (EDA) — Vehicle Casting Defect Detection

**Project Goal:** Automated visual inspection of cast metal components (submersible pump impellers) to detect manufacturing defects.

**Dataset:** Kaggle "Casting product image data for quality inspection" (ravirajsinh45).
- Images: ~300x300 grayscale JPEG captures.
- Classes: `def_front` (Defective casting) vs. `ok_front` (Flawless casting).

### Objectives of this Notebook:
1. Verify dataset structure and directory layout.
2. Analyze **class distribution** (train & test splits) to check for imbalance.
3. Inspect **sample images** from each class to understand visual defects (cracks, pinholes, blemishes).
4. Validate **image dimensions & aspect ratios**.
5. Analyze **pixel intensity distributions** (histogram/KDE) to discover contrast characteristics.
6. Save all charts to `reports/figures/` for documentation.

In [ ]:
# Imports
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Plotting configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

# Directories
# Resolve relative to project root
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data" / "casting_data" / "casting_data"
FIGURES_DIR = BASE_DIR / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project base:   {BASE_DIR}")
print(f"Dataset path:   {DATA_DIR}")
print(f"Figures export: {FIGURES_DIR}")

## 1. Dataset Directory Verification

In [ ]:
train_dir = DATA_DIR / "train"
test_dir = DATA_DIR / "test"

dataset_available = train_dir.exists() and test_dir.exists()

if dataset_available:
    print("Dataset directories found:")
    for path in [train_dir / "def_front", train_dir / "ok_front", test_dir / "def_front", test_dir / "ok_front"]:
        print(f"  ✓ {path.relative_to(BASE_DIR)}: {len(list(path.glob('*.jpeg')) + list(path.glob('*.jpg')))} images")
else:
    print("⚠️ Dataset not detected at expected path:", DATA_DIR)
    print("Please download from Kaggle ('ravirajsinh45/real-life-industrial-dataset-of-casting-product')")
    print("and extract to: data/casting_data/casting_data/")

## 2. Class Balance Analysis

Understanding class distribution prevents false high-accuracy traps and guides our metric choice (e.g., Recall / F1-Score for defect detection).

In [ ]:
def get_class_counts(data_root):
    records = []
    for split in ["train", "test"]:
        for label in ["def_front", "ok_front"]:
            class_name = "Defective" if "def" in label else "OK"
            folder = data_root / split / label
            if folder.exists():
                count = len(list(folder.glob("*.jpeg")) + list(folder.glob("*.jpg")) + list(folder.glob("*.png")))
            else:
                count = 0
            records.append({"Split": split.capitalize(), "Class": class_name, "Folder": label, "Count": count})
    return pd.DataFrame(records)

if dataset_available:
    df_counts = get_class_counts(DATA_DIR)
else:
    # Synthetic placeholder matching Kaggle dataset ground truth
    df_counts = pd.DataFrame([
        {"Split": "Train", "Class": "Defective", "Folder": "def_front", "Count": 3758},
        {"Split": "Train", "Class": "OK", "Folder": "ok_front", "Count": 2875},
        {"Split": "Test", "Class": "Defective", "Folder": "def_front", "Count": 453},
        {"Split": "Test", "Class": "OK", "Folder": "ok_front", "Count": 262},
    ])
    print("[Notice: Displaying standard dataset counts pending local file placement]")

display(df_counts)

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Train split
train_subset = df_counts[df_counts["Split"] == "Train"]
sns.barplot(data=train_subset, x="Class", y="Count", palette=["#e74c3c", "#2ecc71"], ax=axes[0])
axes[0].set_title("Training Set: Class Distribution", fontweight="bold", fontsize=13)
axes[0].set_ylabel("Number of Images")
for p in axes[0].patches:
    height = int(p.get_height())
    axes[0].annotate(f"{height}\n({height/train_subset['Count'].sum():.1%})", 
                    (p.get_x() + p.get_width() / 2., height / 2),
                    ha='center', va='center', color='white', fontweight='bold', fontsize=11)

# Test split
test_subset = df_counts[df_counts["Split"] == "Test"]
sns.barplot(data=test_subset, x="Class", y="Count", palette=["#e74c3c", "#2ecc71"], ax=axes[1])
axes[1].set_title("Test Set: Class Distribution", fontweight="bold", fontsize=13)
axes[1].set_ylabel("Number of Images")
for p in axes[1].patches:
    height = int(p.get_height())
    axes[1].annotate(f"{height}\n({height/test_subset['Count'].sum():.1%})", 
                    (p.get_x() + p.get_width() / 2., height / 2),
                    ha='center', va='center', color='white', fontweight='bold', fontsize=11)

plt.suptitle("Casting Product Dataset — Class Balance Overview", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
fig_path = FIGURES_DIR / "class_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Figure saved to: {fig_path}")
plt.show()

## 3. Visual Inspection: Sample Images per Class

Let's visualize sample images from both categories (`def_front` and `ok_front`) to identify how casting defects visually present on the impeller.

In [ ]:
def get_sample_images(data_root, n_samples=4):
    samples = {"Defective": [], "OK": []}
    def_files = list((data_root / "train" / "def_front").glob("*.jpeg"))[:n_samples]
    ok_files = list((data_root / "train" / "ok_front").glob("*.jpeg"))[:n_samples]
    
    for f in def_files:
        samples["Defective"].append(Image.open(f).convert("RGB"))
    for f in ok_files:
        samples["OK"].append(Image.open(f).convert("RGB"))
    return samples

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

if dataset_available:
    sample_dict = get_sample_images(DATA_DIR, n_samples=4)
    for i, img in enumerate(sample_dict["Defective"]):
        axes[0, i].imshow(img)
        axes[0, i].set_title(f"Defective #{i+1}", color="#c0392b", fontweight="bold")
        axes[0, i].axis("off")
    for i, img in enumerate(sample_dict["OK"]):
        axes[1, i].imshow(img)
        axes[1, i].set_title(f"OK #{i+1}", color="#27ae60", fontweight="bold")
        axes[1, i].axis("off")
else:
    # Synthetic representation demonstrating grid layout
    for i in range(4):
        synthetic_def = np.full((300, 300, 3), 140, dtype=np.uint8)
        cv2.circle(synthetic_def, (150, 150), 90, (180, 180, 180), -1)
        cv2.line(synthetic_def, (120, 130), (145, 170), (40, 40, 40), 3)  # Simulated crack
        axes[0, i].imshow(synthetic_def)
        axes[0, i].set_title(f"Defective #{i+1} (Demo)", color="#c0392b", fontweight="bold")
        axes[0, i].axis("off")
        
        synthetic_ok = np.full((300, 300, 3), 140, dtype=np.uint8)
        cv2.circle(synthetic_ok, (150, 150), 90, (180, 180, 180), -1)
        axes[1, i].imshow(synthetic_ok)
        axes[1, i].set_title(f"OK #{i+1} (Demo)", color="#27ae60", fontweight="bold")
        axes[1, i].axis("off")

axes[0, 0].set_ylabel("DEFECTIVE", fontsize=13, fontweight="bold", color="#c0392b")
axes[1, 0].set_ylabel("OK / NORMAL", fontsize=13, fontweight="bold", color="#27ae60")
plt.suptitle("Sample Inspection: Defective (Top) vs. OK (Bottom)", fontsize=15, fontweight="bold", y=0.98)
plt.tight_layout()
fig_path = FIGURES_DIR / "sample_images.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Figure saved to: {fig_path}")
plt.show()

## 4. Image Dimensions & Aspect Ratio Analysis

Ensure all images share uniform resolution and channel configuration before model ingestion.

In [ ]:
image_metadata = []

if dataset_available:
    # Sample up to 100 images per class for fast dimension checking
    for split in ["train", "test"]:
        for label in ["def_front", "ok_front"]:
            folder = DATA_DIR / split / label
            files = (list(folder.glob("*.jpeg")) + list(folder.glob("*.jpg")))[:100]
            for f in files:
                with Image.open(f) as img:
                    w, h = img.size
                    image_metadata.append({
                        "Split": split,
                        "Class": "Defective" if "def" in label else "OK",
                        "Width": w,
                        "Height": h,
                        "Aspect_Ratio": round(w / h, 3),
                        "Mode": img.mode
                    })
    df_meta = pd.DataFrame(image_metadata)
else:
    # Synthetic representation of expected 300x300 images
    df_meta = pd.DataFrame([{
        "Split": "Train", "Class": "Defective", "Width": 300, "Height": 300, "Aspect_Ratio": 1.0, "Mode": "L"
    }] * 50 + [{
        "Split": "Train", "Class": "OK", "Width": 300, "Height": 300, "Aspect_Ratio": 1.0, "Mode": "L"
    }] * 50)

print("Dimensions Summary:")
display(df_meta[["Width", "Height", "Aspect_Ratio"]].describe())

# Scatter plot of Height vs Width
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=df_meta, x="Width", y="Height", hue="Class", palette=["#e74c3c", "#2ecc71"], alpha=0.7, ax=ax)
ax.set_title("Image Dimensions Distribution (Width vs Height)", fontweight="bold", fontsize=12)
ax.set_xlabel("Width (px)")
ax.set_ylabel("Height (px)")
plt.tight_layout()
fig_path = FIGURES_DIR / "image_size_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Figure saved to: {fig_path}")
plt.show()

## 5. Pixel-Intensity Distribution Analysis

Comparing pixel intensity profiles between OK and Defective parts shows whether anomalies cause localized shadow dips or overall luminance shifts.

In [ ]:
def collect_pixel_intensities(data_root, n_images=50):
    def_pixels = []
    ok_pixels = []
    
    def_files = (list((data_root / "train" / "def_front").glob("*.jpeg")) + 
                 list((data_root / "train" / "def_front").glob("*.jpg")))[:n_images]
    ok_files = (list((data_root / "train" / "ok_front").glob("*.jpeg")) + 
                list((data_root / "train" / "ok_front").glob("*.jpg")))[:n_images]
    
    for f in def_files:
        arr = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if arr is not None:
            def_pixels.extend(arr.flatten()[::20])  # Subsample for plotting efficiency
    for f in ok_files:
        arr = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if arr is not None:
            ok_pixels.extend(arr.flatten()[::20])
            
    return np.array(def_pixels), np.array(ok_pixels)

if dataset_available:
    def_pixels, ok_pixels = collect_pixel_intensities(DATA_DIR, n_images=50)
else:
    # Synthetic Gaussian distributions simulating casting intensities
    np.random.seed(42)
    def_pixels = np.clip(np.random.normal(loc=135, scale=45, size=20000), 0, 255).astype(np.uint8)
    ok_pixels = np.clip(np.random.normal(loc=142, scale=38, size=20000), 0, 255).astype(np.uint8)

fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(def_pixels, label="Defective (def_front)", color="#e74c3c", fill=True, alpha=0.3, lw=2, ax=ax)
sns.kdeplot(ok_pixels, label="OK (ok_front)", color="#2ecc71", fill=True, alpha=0.3, lw=2, ax=ax)

ax.set_title("Pixel Intensity Distribution (KDE): Defective vs. OK Parts", fontweight="bold", fontsize=13)
ax.set_xlabel("Pixel Grayscale Intensity (0 = Black, 255 = White)", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_xlim(0, 255)
ax.legend(fontsize=11)

plt.tight_layout()
fig_path = FIGURES_DIR / "pixel_intensity_distribution.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Figure saved to: {fig_path}")
plt.show()

## 6. Key Takeaways & Pipeline Implications

| Observation | Engineering Implication |
| :--- | :--- |
| **Slight Class Imbalance** | ~57% Defective vs 43% OK. Stratified validation split (`val_split=0.15`) is used. Monitor Recall and F1-Score rather than pure Accuracy. |
| **Square Resolution (~300x300)** | Clean 1:1 aspect ratio. Resizing to `(224, 224)` preserves geometry without stretching. |
| **Monochrome Castings** | Grayscale casting parts. For pretrained CNNs (ImageNet backbones), replicate 1 channel to 3 channels (`color_mode='rgb'`) to retain compatibility with pretrained feature detectors. |
| **Subtle Crack / Void Features** | Data augmentation (rotations, brightness, slight zooms) is essential so the model generalizes across minor part orientation changes and lighting drifts. |
| **Spatial Localization** | Grad-CAM will effectively localize localized anomalies (voids, cracks, pinholes) without needing bounding-box supervision. |